# 강판 결함 분류 — 3단계: 기존 2모델 스태킹의 메타모델 개선

**기준:** 이전 2단계 검증에서 기존 ET+LightGBM 스태킹 0.8277, CatBoost 추가 스태킹 0.8246.  
**실험:** 기본 모델 ET와 LightGBM은 **그대로** 두고 최종 분류기(메타모델)만 바꿉니다. 

- 이전 단계 노트북과 **동일한 개발 데이터 80%**, **외부 3-Fold(random_state=2027)** 를 사용합니다.
- ET/LGB 내부 OOF 확률을 **외부 학습 Fold에서만** 생성하므로 외부 검증 누수를 막습니다.
- 메타모델 후보는 같은 확률 데이터에서 비교해서 실행 시간을 줄입니다.
- **따로 실행 가능**: 이전 노트북을 실행할 필요 없습니다. `Faults.NNA`, `Faults27x7_var`만 같은 폴더에 두세요.
- 이번 외부 Fold로 가장 높은 후보를 선택하면 선택 편향이 생길 수 있으므로, **최종 결정은 새로운 시드의 교차검증**으로 재확인해야 합니다. 최종 테스트셋은 건드리지 않습니다.

In [1]:
# 첫 실행 전에 필요한 패키지만 설치하세요 (미설치 시 주석 해제 후 실행)
# %pip install lightgbm scikit-learn pandas numpy

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, RepeatedStratifiedKFold, cross_val_predict
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import f1_score, classification_report
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore', category=FutureWarning)
RANDOM_STATE = 42
N_SPLITS = 3
N_REPEATS = 1   # 첫 실험은 1회; 모델을 비교할 때 이후 새 검증 시드로 확인
root = Path('.')

for name in ['Faults.NNA', 'Faults27x7_var']:
    if not (root / name).exists():
        raise FileNotFoundError(f'{name} 파일을 이 노트북과 같은 폴더에 넣어 주세요.')

cols = [s.strip() for s in (root / 'Faults27x7_var').read_text(encoding='utf-8').splitlines() if s.strip()]
raw = pd.read_csv(root / 'Faults.NNA', sep=r'\s+', header=None, names=cols)
assert raw.shape == (1941, 34), f'데이터 크기 점검 필요: {raw.shape}'
labels = list(raw.columns[27:34])
encoder = LabelEncoder()
y_full = encoder.fit_transform(raw[labels].idxmax(axis=1))
X_full = raw.iloc[:, :27].drop(columns='TypeOfSteel_A300')
idx_dev, idx_test = train_test_split(
    np.arange(len(y_full)), test_size=0.2, stratify=y_full, random_state=RANDOM_STATE
)
X_dev = X_full.iloc[idx_dev].reset_index(drop=True)
y_dev = y_full[idx_dev]
# 최종 20% 테스트셋은 의도적으로 이번 실험에서 사용하지 않습니다.
print('개발 데이터:', X_dev.shape, '| 따로 보관한 테스트 행 수:', len(idx_test))
print('클래스:', list(encoder.classes_))

개발 데이터: (1552, 26) | 따로 보관한 테스트 행 수: 389
클래스: ['Bumps', 'Dirtiness', 'K_Scatch', 'Other_Faults', 'Pastry', 'Stains', 'Z_Scratch']


In [2]:
# 앞선 1~2단계 노트북과 동일한 베이스 모델 파라미터
et = ExtraTreesClassifier(
    random_state=42, n_jobs=1, n_estimators=500, min_samples_split=5,
    min_samples_leaf=1, max_features=0.8, max_depth=None,
    class_weight='balanced_subsample'
)
lgb = LGBMClassifier(
    objective='multiclass', random_state=42, n_jobs=1, verbose=-1,
    subsample_freq=1, subsample=0.8, reg_lambda=0,
    num_leaves=22, n_estimators=700, min_child_samples=5,
    max_depth=-1, learning_rate=0.05, colsample_bytree=0.6,
    class_weight=None
)

# 메타모델은 ET + LightGBM이 출력한 7개 클래스 확률 × 2 = 14개 값을 입력받습니다.
# '기존 LR C=1'은 이전 결과 0.8277 재현용 기준입니다.
meta_candidates = {
    '기존 LR C=1': LogisticRegression(max_iter=3000, C=1.0),
    'LR C=0.1': LogisticRegression(max_iter=3000, C=0.1),
    'LR C=0.3': LogisticRegression(max_iter=3000, C=0.3),
    'LR C=3': LogisticRegression(max_iter=3000, C=3.0),
    'LR C=10': LogisticRegression(max_iter=3000, C=10.0),
    'LR balanced C=1': LogisticRegression(max_iter=3000, C=1.0, class_weight='balanced'),
    'SVC RBF C=1': make_pipeline(StandardScaler(), SVC(C=1.0, kernel='rbf', gamma='scale')),
    'SVC RBF C=3': make_pipeline(StandardScaler(), SVC(C=3.0, kernel='rbf', gamma='scale')),
    'HGB shallow': HistGradientBoostingClassifier(
        max_iter=100, max_leaf_nodes=7, max_depth=3,
        min_samples_leaf=25, l2_regularization=5,
        learning_rate=0.05, random_state=42
    ),
}

def macro(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro', zero_division=0)

print('메타모델 후보:', len(meta_candidates), '개')

메타모델 후보: 9 개


In [3]:
# 3단계 실험: 외부 Fold의 검증 정답을 활용한 하이퍼파라미터 선택은 아직 하지 않습니다.
# 각 외부 Fold의 학습 데이터에서만 ET/LGB OOF 확률을 만들고, 검증 데이터는 한 번만 예측합니다.
outer = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=2027)
rows = []
pooled = {name: {'y_true': [], 'y_pred': []} for name in meta_candidates}

for fold, (tr, va) in enumerate(outer.split(X_dev, y_dev), 1):
    Xa, Xv = X_dev.iloc[tr], X_dev.iloc[va]
    ya, yv = y_dev[tr], y_dev[va]
    inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

    print(f'외부 Fold {fold}: OOF 확률 생성 중...', flush=True)
    et_oof = cross_val_predict(clone(et), Xa, ya, cv=inner, method='predict_proba', n_jobs=1)
    lgb_oof = cross_val_predict(clone(lgb), Xa, ya, cv=inner, method='predict_proba', n_jobs=1)
    meta_train = np.hstack([et_oof, lgb_oof])

    # 외부 Fold의 학습 데이터 전체로 베이스 모델을 다시 학습해 검증 확률 생성
    et_val = clone(et).fit(Xa, ya).predict_proba(Xv)
    lgb_val = clone(lgb).fit(Xa, ya).predict_proba(Xv)
    meta_valid = np.hstack([et_val, lgb_val])

    fold_row = {'Fold': fold}
    for name, model in meta_candidates.items():
        fitted = clone(model).fit(meta_train, ya)
        pred = fitted.predict(meta_valid)
        fold_row[name] = macro(yv, pred)
        pooled[name]['y_true'].append(np.asarray(yv))
        pooled[name]['y_pred'].append(np.asarray(pred))

    rows.append(fold_row)
    print(f'외부 Fold {fold} 완료 | 기존={fold_row["기존 LR C=1"]:.4f} | '
          f'현재 최고={max((fold_row[n], n) for n in meta_candidates)[1]}', flush=True)

stage3 = pd.DataFrame(rows)
cols = list(meta_candidates)
summary3 = pd.DataFrame({
    '평균 Macro F1': stage3[cols].mean(),
    '표준편차': stage3[cols].std(ddof=0)
}).sort_values('평균 Macro F1', ascending=False)

print('\n[Fold별 성능]')
display(stage3.round(4))
print('\n[메타모델별 평균 성능 — 높은 순]')
display(summary3.round(4))

baseline = float(summary3.loc['기존 LR C=1', '평균 Macro F1'])
best_name = summary3.index[0]
best = float(summary3.iloc[0]['평균 Macro F1'])
print(f'\n기존 LR C=1: {baseline:.4f} (앞선 실험 0.8277과 거의 일치해야 함)')
print(f'이번 실험 1위: {best_name} / {best:.4f} / 기존 대비 {best-baseline:+.4f}')
print('목표 0.835:', '이번 실험에서 도달 — 독립 검증 필요' if best >= .835 else '아직 미달성')

# 결과 기록 (원하면 저장)
stage3.to_csv('stage3_fold_scores.csv', index=False, encoding='utf-8-sig')
summary3.to_csv('stage3_meta_summary.csv', encoding='utf-8-sig')

외부 Fold 1: OOF 확률 생성 중...
외부 Fold 1 완료 | 기존=0.8238 | 현재 최고=LR C=3
외부 Fold 2: OOF 확률 생성 중...
외부 Fold 2 완료 | 기존=0.8193 | 현재 최고=HGB shallow
외부 Fold 3: OOF 확률 생성 중...
외부 Fold 3 완료 | 기존=0.8401 | 현재 최고=LR C=0.3

[Fold별 성능]


,Fold,기존 LR C=1,LR C=0.1,LR C=0.3,LR C=3,LR C=10,LR balanced C=1,SVC RBF C=1,SVC RBF C=3,HGB shallow
0,1,0.8238,0.7990,0.8139,0.8301,0.8223,0.8269,0.8224,0.8161,0.8219
1,2,0.8193,0.8185,0.8095,0.8190,0.8141,0.8215,0.8187,0.8187,0.8248
2,3,0.8401,0.8287,0.8442,0.8288,0.8246,0.8132,0.8228,0.8253,0.8368



[메타모델별 평균 성능 — 높은 순]


,평균 Macro F1,표준편차
HGB shallow,0.8278,0.0065
기존 LR C=1,0.8277,0.0090
LR C=3,0.8260,0.0050
LR C=0.3,0.8226,0.0154
SVC RBF C=1,0.8213,0.0019
LR balanced C=1,0.8205,0.0056
LR C=10,0.8203,0.0045
SVC RBF C=3,0.8200,0.0039
LR C=0.1,0.8154,0.0123



기존 LR C=1: 0.8277 (앞선 실험 0.8277과 거의 일치해야 함)
이번 실험 1위: HGB shallow / 0.8278 / 기존 대비 +0.0001
목표 0.835: 아직 미달성


## 해석 및 다음 단계

1. 먼저 **기존 LR C=1**이 약 **0.8277**로 재현되는지 확인하세요. 다르면 라이브러리 버전이나 데이터 분할을 점검해야 합니다.
2. 이번 표는 **같은 외부 3-Fold를 보며 여러 후보를 비교한 탐색 결과**입니다. 최고 점수가 0.835를 넘더라도 아직 최종 성능이 아닙니다.
3. 높은 후보가 나오면 **다른 시드(예: 2031)의 외부 CV로 후보를 사전 고정하여 재평가**해야 합니다. 반복 검증에서 안정적일 때만 최종 모델 후보로 채택하세요.
4. 최종 테스트 데이터는 모델 및 파라미터 선택이 끝난 뒤 한 번만 평가하는 것이 바람직합니다.

**화면에서 `[메타모델별 평균 성능]` 표를 보여주세요.** 그 결과를 보고 다음 단계 실험을 설계하겠습니다.

In [6]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

# 기존 스태킹의 외부 CV 예측 결과
y_true = np.concatenate(
    pooled['기존 LR C=1']['y_true']
)
y_pred = np.concatenate(
    pooled['기존 LR C=1']['y_pred']
)

names = encoder.classes_

print("[클래스별 성능]")
print(classification_report(
    y_true,
    y_pred,
    labels=np.arange(len(names)),
    target_names=names,
    digits=4,
    zero_division=0
))

print("\n[오분류 행렬]")
cm = pd.DataFrame(
    confusion_matrix(
        y_true,
        y_pred,
        labels=np.arange(len(names))
    ),
    index=names,
    columns=names
)
display(cm)

[클래스별 성능]
              precision    recall  f1-score   support

       Bumps     0.7013    0.6511    0.6753       321
   Dirtiness     0.9722    0.7955    0.8750        44
    K_Scatch     0.9741    0.9617    0.9678       313
Other_Faults     0.7131    0.7993    0.7537       538
      Pastry     0.7426    0.5952    0.6608       126
      Stains     0.9483    0.9483    0.9483        58
   Z_Scratch     0.9320    0.9013    0.9164       152

    accuracy                         0.8003      1552
   macro avg     0.8548    0.8075    0.8282      1552
weighted avg     0.8033    0.8003    0.7998      1552


[오분류 행렬]


,Bumps,Dirtiness,K_Scatch,Other_Faults,Pastry,Stains,Z_Scratch
Bumps,209,0,0,103,6,1,2
Dirtiness,2,35,0,4,3,0,0
K_Scatch,1,0,301,11,0,0,0
Other_Faults,75,1,6,430,17,2,7
Pastry,10,0,0,40,75,0,1
Stains,0,0,0,3,0,55,0
Z_Scratch,1,0,2,12,0,0,137
